In [1]:
import torch
from accelerate import Accelerator
from torch.utils.data import DataLoader
from torchvision import transforms as tf
from torchvision.datasets import FashionMNIST
from torchvision.utils import make_grid, save_image
from torch_ema import ExponentialMovingAverage as EMA
from tqdm import tqdm
from IPython.display import display
import matplotlib.pyplot as plt

from smalldiffusion import (
    ScheduleLogLinear, samples, training_loop, MappedDataset, Unet, Scaled,
    img_train_transform, img_normalize
)

c:\Users\dzluk\Syrinx\audio-diffusion-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_batch_size = 128
sample_batch_size = 8
epochs = 10

# Setup
a = Accelerator()

# This is the img_train_transform from smalldiffusion, with an additional resize
resize_transform = tf.Compose([
    tf.Resize((64, 64)),          # Upscale to 64x64
    tf.RandomHorizontalFlip(),
    tf.ToTensor(),
    tf.Lambda(lambda t: (t * 2) - 1)
])
dataset = MappedDataset(FashionMNIST('datasets', train=True, download=True, transform=resize_transform), lambda x: x[0])
loader = DataLoader(dataset, batch_size=train_batch_size, shuffle=True)
schedule = ScheduleLogLinear(sigma_min=0.01, sigma_max=20, N=800)
model = Scaled(Unet)(28, 1, 1, ch=64, ch_mult=(1, 1, 2), attn_resolutions=(14,))

In [9]:
# Load pre-trained model (optional - skip to Train cell to train from scratch)
checkpoint_path = 'fashion-checkpoint.pth'

import os
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=a.device))
    print(f"Loaded model from {checkpoint_path}")
else:
    print(f"No checkpoint found at {checkpoint_path}, you should train from scratch")

Loaded model from fashion-checkpoint.pth


In [ ]:
# Train
# ema = EMA(model.parameters(), decay=0.999).to(a.device)
losses = []
for ns in training_loop(loader, model, schedule, epochs=epochs, lr=7e-4, accelerator=a):
    losses.append(ns.loss.item())
    ns.pbar.set_description(f'Loss={ns.loss.item():.5}')
    # ema.update()

# Save checkpoint
torch.save(model.state_dict(), 'fashion-checkpoint.pth')

# Plot training losses
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

In [11]:
# Sample
# with ema.average_parameters():
*xt, x0 = samples(model, schedule.sample_sigmas(20), gam=1.6,
                    batchsize=sample_batch_size, accelerator=a)
grid = img_normalize(make_grid(x0))
    
# Display in notebook
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis('off')
plt.title('Generated Fashion MNIST Samples')
plt.show()

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)